In [1]:
import random
import numpy as np
import pandas as pd
import re
import string
import tensorflow as tf
from tensorflow.keras.layers import LSTM, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.layers import TextVectorization

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
from zipfile import ZipFile

path_zip = "/content/contradictory-my-dear-watson.zip"

with ZipFile(path_zip, 'r') as zf:
  zf.extractall()

In [4]:
root = "./"

train_df = pd.read_csv(root + "train.csv")
test_df = pd.read_csv(root + "test.csv")

In [5]:
train_df.drop(["id", "lang_abv", "language"],axis=1,inplace=True)
test_df.drop(["id", "lang_abv", "language"],axis=1,inplace=True)

In [6]:
max_len = max(len(text.split()) for text in train_df["premise"])
print(f'Max length of premise: {max_len}')

vectorizer = TextVectorization(output_mode='int', split='whitespace',
                               standardize='strip_punctuation',
                               max_tokens=10000,
                               output_sequence_length=max_len)

Max length of premise: 196


In [7]:
def make_dataset(premise, hypothesis, label,
                 batch=32, buffer=10000):

    train_dataset = tf.data.Dataset.from_tensor_slices(((premise, hypothesis), label))
    dataset = train_dataset.batch(batch).shuffle(buffer).prefetch(tf.data.AUTOTUNE)

    val_size = int(len(dataset) * 0.2)

    # Split the dataset
    val_ds = dataset.take(val_size)
    train_ds = dataset.skip(val_size)

    return train_ds, val_ds

In [8]:
def process_text(text):
    text = text.lower()

    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)

    text = re.sub(r'\d+', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
premise = train_df["premise"].apply(process_text)
hypothesis = train_df["hypothesis"].apply(process_text)
label = train_df["label"].to_numpy()

In [10]:
vectorizer.adapt(np.concatenate([premise, hypothesis]))
vocab_size = vectorizer.vocabulary_size()
print(f'Vocabulary size: {vocab_size}')

Vocabulary size: 10000


In [11]:
premise = vectorizer(premise)
hypothesis = vectorizer(hypothesis)

In [12]:
train_ds, val_ds = make_dataset(premise, hypothesis, label)

In [13]:
print(train_ds.element_spec)
print(val_ds.element_spec)

((TensorSpec(shape=(None, 196), dtype=tf.int64, name=None), TensorSpec(shape=(None, 196), dtype=tf.int64, name=None)), TensorSpec(shape=(None,), dtype=tf.int64, name=None))
((TensorSpec(shape=(None, 196), dtype=tf.int64, name=None), TensorSpec(shape=(None, 196), dtype=tf.int64, name=None)), TensorSpec(shape=(None,), dtype=tf.int64, name=None))


In [14]:
def get_similarity(premise, hypothesis):
    premise = tf.cast(premise, dtype=tf.float32)
    premise = premise / tf.norm(premise, axis=-1, keepdims=True)

    hypothesis = tf.cast(hypothesis, dtype=tf.float32)
    hypothesis = hypothesis / tf.norm(hypothesis, axis=-1, keepdims=True)

    similarity = tf.reduce_sum(premise * hypothesis, axis=0)

    return similarity

In [15]:
class Base_Model(Model):
    def __init__(self, vocab_size, embedding_dim, name="Base_Model"):
        super(Base_Model, self).__init__(name=name)

        # Definindo as camadas do modelo
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        self.rnn1 = tf.keras.layers.Bidirectional(LSTM(64, return_sequences=True))
        self.batch_norm1 = BatchNormalization()

        self.rnn2 = tf.keras.layers.Bidirectional(LSTM(128, return_sequences=True))
        self.batch_norm2 = BatchNormalization()

        self.rnn3 = tf.keras.layers.Bidirectional(LSTM(256, return_sequences=True))
        self.batch_norm3 = BatchNormalization()

    def call(self, inputs, training=None):
        x = tf.cast(inputs, dtype=tf.int64)

        x = self.embedding(x)

        x = self.rnn1(x)
        x = self.batch_norm1(x, training=training)

        x = self.rnn2(x)
        x = self.batch_norm2(x, training=training)

        x = self.rnn3(x)
        x = self.batch_norm3(x, training=training)

        x = tf.reduce_mean(x, axis=1)

        return x

In [42]:
class Siameses(Model):
    def __init__(self, vocab_size, embedding_dim, name="Siameses"):
        super(Siameses, self).__init__(name=name)

        self.shared_model = Base_Model(vocab_size, embedding_dim)
        self.dense = tf.keras.layers.Dense(3, activation='softmax')

    def call(self, inputs):
        premise, hypothesis = inputs

        premise_output = self.shared_model(premise)
        hypothesis_output = self.shared_model(hypothesis)

        similarity = get_similarity(premise_output, hypothesis_output)

        # Reshape similarity to be a 2D tensor
        # similarity = tf.expand_dims(similarity, axis=0)

        output = self.dense(similarity)

        return output

In [43]:
embedding_dim = 128
model = Siameses(vocab_size=vocab_size, embedding_dim=embedding_dim)

input_a = tf.keras.Input(shape=(196,), dtype="int64")
input_b = tf.keras.Input(shape=(196,), dtype="int64")
model([input_a, input_b])

tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:1331: UserWarning: Layer 'Siameses' looks like it has unbuilt state, but Keras is not able to trace the layer `call()` in order to build it automatically. Possible causes:
1. The `call()` method of your layer may be crashing. Try to `__call__()` the layer eagerly on some test input first to see if it works. E.g. `x = np.random.random((3, 4)); y = layer(x)`
2. If the `call()` method is correct, then you may need to implement the `def build(self, input_shape)` method on your layer. It should create all variables used by the layer (e.g. by calling `layer.build()` on all its children layers).
Exception encountered: ''Input 0 of layer "dense_2" is incompatible with the layer: expected min_ndim=2, found ndim=1. Full shape received: (512,)''
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:372: UserWarning: `build()` was called on layer 'Siameses', however the layer does not have a `build()` me

ValueError: Exception encountered when calling Siameses.call().

[1mCould not automatically infer the output shape / dtype of 'Siameses' (of type Siameses). Either the `Siameses.call()` method is incorrect, or you need to implement the `Siameses.compute_output_spec() / compute_output_shape()` method. Error encountered:

Input 0 of layer "dense_2" is incompatible with the layer: expected min_ndim=2, found ndim=1. Full shape received: (512,)[0m

Arguments received by Siameses.call():
  • args=(['<KerasTensor shape=(None, 196), dtype=int64, sparse=None, name=keras_tensor_32>', '<KerasTensor shape=(None, 196), dtype=int64, sparse=None, name=keras_tensor_33>'],)
  • kwargs=<class 'inspect._empty'>

In [40]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['accuracy'])

In [24]:
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# loss = tf.keras.losses.SparseCategoricalCrossentropy()
# model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

In [41]:
# Train the model
history = model.fit(train_ds, validation_data=val_ds, epochs=30)

Epoch 1/30


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/compile_loss/mean_squared_error/sub/BroadcastGradientArgs defined at (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code

  File "/usr/local/lib/python3.10/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.py", line 619, in start

  File "/usr/local/lib/python3.10/dist-packages/tornado/platform/asyncio.py", line 195, in start

  File "/usr/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/usr/lib/python3.10/asyncio/base_events.py", line 1909, in _run_once

  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run

  File "/usr/local/lib/python3.10/dist-packages/tornado/ioloop.py", lin

In [ ]:
test_premise = test_df["premise"].apply(process_text)
test_hypothesis = test_df["hypothesis"].apply(process_text)

test_premise = vectorizer(test_premise)
test_hypothesis = vectorizer(test_hypothesis)

submission = pd.read_csv(root + "sample_submission.csv")
y_test = submission["prediction"].values
y_test = y_test.reshape(-1, 1)

In [ ]:
y_pred = model.predict([test_premise, test_hypothesis])
y_pred = np.argmax(y_pred, axis=1)

In [ ]:
acc = np.mean(y_pred == y_test)
print(f"Accuracy: {acc}")